# MVP03 — Camada Bronze

Lê as tabelas da camada `staging` e as persiste na camada `bronze`, acrescentando
metadados de processamento.

**Posição no pipeline:** terceiro notebook. Requer `MVP02-Download` executado.

**Princípio da camada:** a Bronze preserva o dado bruto. Nenhuma transformação de
conteúdo é aplicada — nem tipagem, nem limpeza, nem filtro. Todas as colunas de negócio
permanecem como texto, exatamente como chegaram da origem. A Bronze existe para que as
camadas seguintes possam ser reprocessadas sem depender de nova coleta no Google Drive.

## 📚 Sobre este Notebook

Este notebook implementa a ingestão da **camada Bronze** a partir dos dados já carregados na **camada Staging**.

### Arquitetura Medalhão

* **📂 Staging**: Dados brutos copiados diretamente do Google Drive (notebook MVP02-Download)
* **🥉 Bronze**: Dados brutos com metadados de rastreabilidade do processamento (este notebook)
* **🥈 Silver**: Dados limpos e padronizados  (notebook MVP04-Silver)
* **🥇 Gold**: Dados agregados e prontos para consumo  (notebook MVP05-Gold)

### 🎯 Objetivo
Ler as tabelas da camada Staging e carregá-las na camada Bronze, adicionando metadados de rastreabilidade:
* `data_carga_bronze`: timestamp de quando os dados foram processados para Bronze
* `origem_tabela`: identifica a tabela de staging de origem

In [0]:
%sql
-- Define o catálogo ativo como mvpDataEngineer no Unity Catalog.
USE CATALOG mvpDataEngineer;

-- Define o schema ativo como bronze dentro do catálogo atual.
USE SCHEMA bronze;

## 🔧 Etapa 1 — Parâmetros

Definição de parâmetros que serão utilizados durante o notebook.

In [0]:
CATALOGO = "mvpdataengineer"

# Mesmas tabelas produzidas pelo MVP02-Download.
TABELAS = ["leitura_pluviometros", "medicao_bacias", "medicao_laje"]

print(f"Catálogo: {CATALOGO}")
print(f"Tabelas a processar: {TABELAS}")

## 🥉 Etapa 2: Ingestão da Staging para Bronze

Lê cada tabela da camada **Staging**, adiciona metadados de rastreabilidade do processamento e grava na camada **Bronze**.

### Metadados adicionados:
* `data_carga_bronze`: timestamp de quando os dados foram processados para Bronze
* `origem_tabela`: identifica a tabela de staging de origem (ex: `staging.leitura_pluviometros`)

### Princípios da camada Bronze:
* ✅ **Preservar dados brutos** - sem transformação de conteúdo
* ✅ **Adicionar rastreabilidade** - metadados de processamento
* ✅ **Formato Delta Lake** - formato otimizado do Databricks
* ✅ **Histórico completo** - possibilita auditoria e reprocessamento

In [0]:
from pyspark.sql.functions import current_timestamp, lit

print("Carga Staging → Bronze\n" + "=" * 70)

resumo = []

for tabela in TABELAS:
    origem  = f"{CATALOGO}.staging.{tabela}"
    destino = f"{CATALOGO}.bronze.{tabela}"

    df_staging = spark.table(origem)
    n_origem = df_staging.count()

    # Guarda: staging vazia indica falha na coleta. Prosseguir apagaria a Bronze.
    if n_origem == 0:
        raise RuntimeError(
            f"A tabela {origem} está vazia. Carga da Bronze abortada para não "
            "sobrescrever dados válidos com um conjunto vazio. "
            "Verifique a execução do notebook MVP02-Download."
        )

    df_bronze = (df_staging
                 .withColumn("data_carga_bronze", current_timestamp())
                 .withColumn("origem_tabela", lit(f"staging.{tabela}")))

    (df_bronze.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(destino))

    n_destino = spark.table(destino).count()

    print(f"\n{tabela}")
    print(f"  Origem  : {origem}  ({n_origem} registros)")
    print(f"  Destino : {destino}  ({n_destino} registros)")
    print(f"  Colunas : {len(df_staging.columns)} + 2 metadados = {len(df_bronze.columns)}")

    resumo.append({
        "tabela": tabela,
        "registros_staging": n_origem,
        "registros_bronze": n_destino,
        "colunas": len(df_bronze.columns),
        "integro": n_origem == n_destino,
    })

print("\n" + "=" * 70)
print("Carga concluída.")
display(spark.createDataFrame(resumo))

## 🔍 Etapa 3: Validação dos Dados Bronze

A Bronze é uma cópia fiel da Staging: a contagem de registros precisa ser idêntica em
todas as tabelas. Divergência indica perda de dados na cópia e interrompe o pipeline.

#### O que verifica?
* **Total de registros** por tabela
* **Última carga** na camada Bronze
* **Amostras** dos dados carregados
* **Colunas de metadados** adicionadas

In [0]:
%sql
-- Resumo das tabelas Bronze: total de registros e última carga
SELECT 
  'LEITURA_PLUVIOMETROS' as tabela,
  COUNT(*) as total_registros,
  MAX(data_carga_bronze) as ultima_carga_bronze,
  MAX(data_ingestao) as ultima_ingestao_staging
FROM bronze.leitura_pluviometros

UNION ALL

SELECT 
  'MEDICAO_BACIAS' as tabela,
  COUNT(*) as total_registros,
  MAX(data_carga_bronze) as ultima_carga_bronze,
  MAX(data_ingestao) as ultima_ingestao_staging
FROM bronze.medicao_bacias

UNION ALL

SELECT 
  'MEDICAO_LAJE' as tabela,
  COUNT(*) as total_registros,
  MAX(data_carga_bronze) as ultima_carga_bronze,
  MAX(data_ingestao) as ultima_ingestao_staging
FROM bronze.medicao_laje

ORDER BY tabela;

In [0]:
%sql
-- Amostra de LEITURA_PLUVIOMETROS (primeiras 5 linhas)
-- Inclui os metadados de staging e bronze
SELECT * FROM mvpDataEngineer.bronze.leitura_pluviometros LIMIT 10;

In [0]:
%sql
-- Amostra de MEDICAO_BACIAS (primeiras 5 linhas)
SELECT * FROM mvpDataEngineer.bronze.medicao_bacias LIMIT 10;

In [0]:
%sql
-- Amostra de MEDICAO_LAJE (primeiras 5 linhas)
SELECT * FROM mvpDataEngineer.bronze.medicao_laje LIMIT 10;

## 📝 Etapa 4: Documentação das Tabelas e Colunas

### Por que documentar?
Comentários no Unity Catalog tornam os dados autodescritivos: qualquer pessoa que consultar as tabelas via Catalog Explorer, SQL ou dashboards verá a descrição de cada coluna sem precisar procurar em documentação externa.

### Como funciona?
Usamos `COMMENT ON TABLE` e `COMMENT ON COLUMN` para registrar descrições diretamente nos metadados do Unity Catalog. Os comentários ficam visíveis em:
* **Catalog Explorer** — descrição de tabelas e colunas
* **Autocomplete do SQL Editor** — tooltip ao digitar nomes de colunas
* **Lineage e Dashboards** — contexto para análise
* **APIs do Unity Catalog** — acesso programático à documentação

In [0]:
%sql
-- =========================================
-- COMENTÁRIOS DA TABELA: leitura_pluviometros
-- =========================================

COMMENT ON TABLE mvpdataengineer.bronze.leitura_pluviometros IS
  'Camada Bronze. Leituras pluviométricas das cinco estações da salina, preservadas como vieram da origem e acrescidas de metadados de processamento. Série histórica desde 2014, registrada por evento e não diariamente.';

COMMENT ON COLUMN mvpdataengineer.bronze.leitura_pluviometros.PROTOCOLO IS 'Identificador da solicitação no formulário de origem. Uma solicitação agrupa várias leituras.';
COMMENT ON COLUMN mvpdataengineer.bronze.leitura_pluviometros.SOLICDATA IS 'Data e hora do lançamento no sistema (dd/MM/yyyy HH:mm:ss). Distinta da data da leitura: um lançamento pode registrar medições retroativas.';
COMMENT ON COLUMN mvpdataengineer.bronze.leitura_pluviometros.STATUS IS 'Situação no fluxo de aprovação do formulário.';
COMMENT ON COLUMN mvpdataengineer.bronze.leitura_pluviometros.UNIDADE IS 'Unidade da empresa à qual a estação pertence.';
COMMENT ON COLUMN mvpdataengineer.bronze.leitura_pluviometros.ESTACAO IS 'Rótulo da estação pluviométrica conforme registrado no formulário. Contém variação de grafia, normalizada na camada Silver.';
COMMENT ON COLUMN mvpdataengineer.bronze.leitura_pluviometros.DATAREGISTRO IS 'Data da leitura em campo (dd/MM/yyyy).';
COMMENT ON COLUMN mvpdataengineer.bronze.leitura_pluviometros.MILIMETROS IS 'Precipitação medida, em milímetros. Decimal com vírgula. Ausência representada por hífen.';
COMMENT ON COLUMN mvpdataengineer.bronze.leitura_pluviometros.OBSERVACOES IS 'Campo livre do formulário. Sem preenchimento nesta fonte.';

-- Metadados de rastreabilidade (Staging)
COMMENT ON COLUMN mvpDataEngineer.bronze.leitura_pluviometros.data_ingestao IS 'Timestamp da ingestão na camada Staging';
COMMENT ON COLUMN mvpDataEngineer.bronze.leitura_pluviometros.origem IS 'Origem dos dados (google_drive)';
COMMENT ON COLUMN mvpDataEngineer.bronze.leitura_pluviometros.arquivo_origem IS 'Nome do arquivo Excel de origem';

-- Metadados de rastreabilidade (Bronze)
COMMENT ON COLUMN mvpDataEngineer.bronze.leitura_pluviometros.data_carga_bronze IS 'Timestamp da carga na camada Bronze';
COMMENT ON COLUMN mvpDataEngineer.bronze.leitura_pluviometros.origem_tabela IS 'Tabela de staging de origem';

In [0]:
%sql
-- =========================================
-- COMENTÁRIOS DA TABELA: medicao_bacias
-- =========================================

COMMENT ON TABLE mvpdataengineer.bronze.medicao_bacias IS
  'Camada Bronze. Medição diária da evaporação nas bacias pelo método de reposição até marca de referência, preservada como veio da origem e acrescida de metadados de processamento.';

COMMENT ON COLUMN mvpdataengineer.bronze.medicao_bacias.PROTOCOLO IS 'Identificador da solicitação no formulário de origem. Uma solicitação agrupa vários dias e os pontos de medição.';
COMMENT ON COLUMN mvpdataengineer.bronze.medicao_bacias.SOLICDATA IS 'Data e hora do lançamento no sistema (dd/MM/yyyy HH:mm:ss). Distinta da data da medição.';
COMMENT ON COLUMN mvpdataengineer.bronze.medicao_bacias.STATUS IS 'Situação no fluxo de aprovação. Registros com status diferente de finalizado não foram homologados pela operação.';
COMMENT ON COLUMN mvpdataengineer.bronze.medicao_bacias.UNIDADE IS 'Unidade da empresa à qual a bacia pertence.';
COMMENT ON COLUMN mvpdataengineer.bronze.medicao_bacias.BACIA IS 'Ponto de medição de evaporação.';
COMMENT ON COLUMN mvpdataengineer.bronze.medicao_bacias.DATAREGISTRO IS 'Data da medição em campo (dd/MM/yyyy).';
COMMENT ON COLUMN mvpdataengineer.bronze.medicao_bacias.AGUADOCE_COLOCADO IS 'Volume de água doce reposto até a marca de referência, em litros. Corresponde à evaporação do dia. Ausência representada por hífen.';
COMMENT ON COLUMN mvpdataengineer.bronze.medicao_bacias.AGUADOCE_RETIRADO IS 'Volume de água doce retirado por excedente de chuva, em litros. Mutuamente exclusivo com AGUADOCE_COLOCADO. Ausência representada por hífen.';
COMMENT ON COLUMN mvpdataengineer.bronze.medicao_bacias.SALMOURA_COLOCADO IS 'Volume de salmoura reposto até a marca de referência, em litros. Corresponde à evaporação do dia. Ausência representada por hífen.';
COMMENT ON COLUMN mvpdataengineer.bronze.medicao_bacias.SALMOURA_RETIRADO IS 'Volume de salmoura retirado por excedente de chuva, em litros. Mutuamente exclusivo com SALMOURA_COLOCADO. Ausência representada por hífen.';
COMMENT ON COLUMN mvpdataengineer.bronze.medicao_bacias.SALMOURA_DENSIDADE IS 'Concentração da salmoura em graus Baumé (°Bé), medida com densímetro. Decimal com vírgula. Ausência representada por hífen.';
COMMENT ON COLUMN mvpdataengineer.bronze.medicao_bacias.OBSERVACOES IS 'Campo livre do formulário. Sem preenchimento nesta fonte.';

-- Metadados de rastreabilidade (Staging)
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_bacias.data_ingestao IS 'Timestamp da ingestão na camada Staging';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_bacias.origem IS 'Origem dos dados (google_drive)';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_bacias.arquivo_origem IS 'Nome do arquivo Excel de origem';

-- Metadados de rastreabilidade (Bronze)
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_bacias.data_carga_bronze IS 'Timestamp da carga na camada Bronze';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_bacias.origem_tabela IS 'Tabela de staging de origem';

In [0]:
%sql
-- =========================================
-- COMENTÁRIOS DA TABELA: medicao_laje
-- =========================================

COMMENT ON TABLE mvpdataengineer.bronze.medicao_laje IS
  'Camada Bronze. Medição mensal da laje de sal dos 25 cristalizadores por meio de 8 réguas, preservada como veio da origem e acrescida de metadados de processamento.';

COMMENT ON COLUMN mvpdataengineer.bronze.medicao_laje.PROTOCOLO IS 'Identificador da solicitação no formulário de origem. Uma solicitação agrupa os cristalizadores de uma campanha de medição.';
COMMENT ON COLUMN mvpdataengineer.bronze.medicao_laje.SOLICDATA IS 'Data e hora do lançamento no sistema (dd/MM/yyyy HH:mm:ss). Distinta da data da medição e usada como critério de desempate entre lançamentos duplicados.';
COMMENT ON COLUMN mvpdataengineer.bronze.medicao_laje.STATUS IS 'Situação no fluxo de aprovação do formulário.';
COMMENT ON COLUMN mvpdataengineer.bronze.medicao_laje.UNIDADE IS 'Unidade da empresa à qual o cristalizador pertence.';
COMMENT ON COLUMN mvpdataengineer.bronze.medicao_laje.DATAMEDICAO IS 'Data da medição em campo (dd/MM/yyyy). O intervalo entre medições consecutivas não é fixo.';
COMMENT ON COLUMN mvpdataengineer.bronze.medicao_laje.CRISTALIZADOR IS 'Identificador do cristalizador medido. Registrado com zero à esquerda na origem.';
COMMENT ON COLUMN mvpdataengineer.bronze.medicao_laje.AREAHECTARES IS 'Área do cristalizador em hectares. Constante por cristalizador: é atributo de dimensão, não medida.';
COMMENT ON COLUMN mvpdataengineer.bronze.medicao_laje.REGUA_1 IS 'Altura livre da régua 1 acima da laje, em centímetros. Régua de 1 metro: 100 é o limite superior da leitura.';
COMMENT ON COLUMN mvpdataengineer.bronze.medicao_laje.REGUA_2 IS 'Altura livre da régua 2 acima da laje, em centímetros. Régua de 1 metro: 100 é o limite superior da leitura.';
COMMENT ON COLUMN mvpdataengineer.bronze.medicao_laje.REGUA_3 IS 'Altura livre da régua 3 acima da laje, em centímetros. Régua de 1 metro: 100 é o limite superior da leitura.';
COMMENT ON COLUMN mvpdataengineer.bronze.medicao_laje.REGUA_4 IS 'Altura livre da régua 4 acima da laje, em centímetros. Régua de 1 metro: 100 é o limite superior da leitura.';
COMMENT ON COLUMN mvpdataengineer.bronze.medicao_laje.REGUA_5 IS 'Altura livre da régua 5 acima da laje, em centímetros. Régua de 1 metro: 100 é o limite superior da leitura.';
COMMENT ON COLUMN mvpdataengineer.bronze.medicao_laje.REGUA_6 IS 'Altura livre da régua 6 acima da laje, em centímetros. Régua de 1 metro: 100 é o limite superior da leitura.';
COMMENT ON COLUMN mvpdataengineer.bronze.medicao_laje.REGUA_7 IS 'Altura livre da régua 7 acima da laje, em centímetros. Régua de 1 metro: 100 é o limite superior da leitura.';
COMMENT ON COLUMN mvpdataengineer.bronze.medicao_laje.REGUA_8 IS 'Altura livre da régua 8 acima da laje, em centímetros. Régua de 1 metro: 100 é o limite superior da leitura.';
COMMENT ON COLUMN mvpdataengineer.bronze.medicao_laje.OBSERVACOES IS 'Campo livre do formulário. Registra eventualmente os dias de fabricação e a ocorrência de colheita.';

-- Metadados de rastreabilidade (Staging)
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_laje.data_ingestao IS 'Timestamp da ingestão na camada Staging';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_laje.origem IS 'Origem dos dados (google_drive)';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_laje.arquivo_origem IS 'Nome do arquivo Excel de origem';

-- Metadados de rastreabilidade (Bronze)
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_laje.data_carga_bronze IS 'Timestamp da carga na camada Bronze';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_laje.origem_tabela IS 'Tabela de staging de origem';